## TC 5033
## Deep Learning
## Transformers

#### Activity 4: Implementing a Translator

- Objective

To understand the Transformer Architecture by Implementing a translator.

- Instructions

    This activity requires submission in teams. While teamwork is encouraged, each member is expected to contribute individually to the assignment. The final submission should feature the best arguments and solutions from each team member. Only one person per team needs to submit the completed work, but it is imperative that the names of all team members are listed in a Markdown cell at the very beginning of the notebook (either the first or second cell). Failure to include all team member names will result in the grade being awarded solely to the individual who submitted the assignment, with zero points given to other team members (no exceptions will be made to this rule).

    Follow the provided code. The code already implements a transformer from scratch as explained in one of [week's 9 videos](https://youtu.be/XefFj4rLHgU)

    Since the provided code already implements a simple translator, your job for this assignment is to understand it fully, and document it using pictures, figures, and markdown cells.  You should test your translator with at least 10 sentences. The dataset used for this task was obtained from [Tatoeba, a large dataset of sentences and translations](https://tatoeba.org/en/downloads).
  
- Evaluation Criteria

    - Code Readability and Comments
    - Traning a translator
    - Translating at least 10 sentences.

- Submission

Submit this Jupyter Notebook in canvas with your complete solution, ensuring your code is well-commented and includes Markdown cells that explain your design choices, results, and any challenges you encountered.



### Team 21: 

* Juan Pablo Carvajal Acosta ------ A01796843
* Luis Manuel Velasco Iglesias ---- A00226599
* Mayra Hernández Alba ------------ A01796828
* Sergio David Jardon Avalos ------ A01688787

### Dataset: Tatoeba English-Spanish
Data obtained from [ManyThings.org](https://www.manythings.org/anki/), 
a preprocessed mirror of the [Tatoeba](https://tatoeba.org) corpus.  
Each line contains: `English [TAB] Spanish [TAB] Attribution`

In [1]:
import os, zipfile

EXTRACT_DIR = "/content/spa_eng/"
PATH = "/content/spa_eng/spa.txt"

if not os.path.exists(PATH):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !wget -q -O /content/spa-eng.zip "https://www.manythings.org/anki/spa-eng.zip"
    with zipfile.ZipFile("/content/spa-eng.zip", 'r') as z:
        z.extractall(EXTRACT_DIR)
    print(f" Dataset ready at: {PATH}")
else:
    print("Already downloaded.")


 Dataset ready at: /content/spa_eng/spa.txt


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math
import numpy as np
import re

torch.manual_seed(23)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


## Transformer - Attention is all you need

The Transformer (Vaswani et al., 2017) processes sequences entirely through
**attention mechanisms**, with no recurrence or convolution, which allows full
parallelization during training.

<p align="center">
  <img src="transformer.png" alt="transformer architecture" style="width:35%; height:auto;">
</p>

**Hyperparameters:**

| Parameter   | Value |
|-------------|-------|
| d_model     | 512   |
| num_heads   | 8     |
| d_ff        | 2048  |
| num_layers  | 6     |
| dropout     | 0.1   |
| MAX_SEQ_LEN | 128   |

In [4]:
MAX_SEQ_LEN = 128

### Positional Embedding
Since self-attention has no built-in sense of word order, positional encodings 
are added to the token embeddings using sinusoidal functions:

- **Even dimensions:** $PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$
- **Odd dimensions:** $PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$

This allows the model to distinguish the position of words in a sequence.

In [5]:
class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len = MAX_SEQ_LEN):
        super().__init__()
        self.pos_embed_matrix = torch.zeros(max_seq_len, d_model, device=device)
        token_pos = torch.arange(0, max_seq_len, dtype = torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() 
                             * (-math.log(10000.0)/d_model))
        self.pos_embed_matrix[:, 0::2] = torch.sin(token_pos * div_term)
        self.pos_embed_matrix[:, 1::2] = torch.cos(token_pos * div_term)
        self.pos_embed_matrix = self.pos_embed_matrix.unsqueeze(0).transpose(0,1)
        
    def forward(self, x):
#         print(self.pos_embed_matrix.shape)
#         print(x.shape)
        return x + self.pos_embed_matrix[:x.size(0), :]

### Multi-Head Attention

<p align="center">
  <img src="multhead.png" alt="multi-head attention" style="width:25%; height:auto;">
</p>


The core of the Transformer. Q (Query), K (Key), V (Value) are linearly projected 
and split into `num_heads=8` parallel heads, each computing:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Dividing by $\sqrt{d_k} = 8$ prevents vanishing gradients from large dot products. 
Results from all heads are concatenated and projected via $W_o$.


In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model = 512, num_heads = 8):
        super().__init__()
        assert d_model % num_heads == 0, 'Embedding size not compatible with num heads'
        
        self.d_v = d_model // num_heads
        self.d_k = self.d_v
        self.num_heads = num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, Q, K, V, mask = None):
        batch_size = Q.size(0)
        '''
        Q, K, V -> [batch_size, seq_len, num_heads*d_k]
        after transpose Q, K, V -> [batch_size, num_heads, seq_len, d_k]
        '''
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        
        weighted_values, attention = self.scale_dot_product(Q, K, V, mask)
        weighted_values = weighted_values.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads*self.d_k)
        weighted_values = self.W_o(weighted_values)
        
        return weighted_values, attention
        
        
    def scale_dot_product(self, Q, K, V, mask = None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention = F.softmax(scores, dim = -1)
        weighted_values = torch.matmul(attention, V)
        
        return weighted_values, attention


### Feed-Forward Network (FFN)
A two-layer MLP applied independently to each position:

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

Expands from $d_{model}=512 \rightarrow d_{ff}=2048 \rightarrow 512$, 
adding non-linear transformation capacity after attention.

In [7]:
class PositionFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        
    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

### Encoder
Stacks 6 `EncoderSubLayer` blocks. Each applies:
1. Self-Attention over all source tokens
2. Add & Norm (residual connection + LayerNorm)
3. FFN
4. Add & Norm

In [8]:
class EncoderSubLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.droupout1 = nn.Dropout(dropout)
        self.droupout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask = None):
        attention_score, _ = self.self_attn(x, x, x, mask)
        x = x + self.droupout1(attention_score)
        x = self.norm1(x)
        x = x + self.droupout2(self.ffn(x))
        return self.norm2(x)

class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

### Decoder
Stacks 6 `DecoderSubLayer` blocks with **three** sub-layers:
1. **Masked Self-Attention** — causal mask prevents seeing future tokens
2. **Cross-Attention** — $Q$ from decoder, $K$/$V$ from encoder output (this is where translation happens)
3. **FFN + Add & Norm**

In [9]:
class DecoderSubLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, target_mask=None, encoder_mask=None):
        attention_score, _ = self.self_attn(x, x, x, target_mask)
        x = x + self.dropout1(attention_score)
        x = self.norm1(x)
        
        encoder_attn, _ = self.cross_attn(x, encoder_output, encoder_output, encoder_mask)
        x = x + self.dropout2(encoder_attn)
        x = self.norm2(x)
        
        ff_output = self.feed_forward(x)
        x = x + self.dropout3(ff_output)
        return self.norm3(x)
        
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([DecoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x, encoder_output, target_mask, encoder_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, target_mask, encoder_mask)
        return self.norm(x)

### Full Transformer Model
Combines encoder and decoder into a single module.
- Token embeddings are scaled by $\sqrt{d_{model}}$ to balance with positional encodings
- The same `PositionalEmbedding` is shared for both source and target
- Final `output_layer` is a Linear layer mapping hidden states → target vocabulary logits

**Masking strategy:**
- `source_mask`: padding mask — ignores `<PAD>` tokens (index 0)
- `target_mask`: padding mask **AND** causal lower-triangular mask — prevents 
  the decoder from attending to future tokens during training


In [10]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers,
                 input_vocab_size, target_vocab_size, 
                 max_len=MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.encoder_embedding = nn.Embedding(input_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model)
        self.pos_embedding = PositionalEmbedding(d_model, max_len)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.output_layer = nn.Linear(d_model, target_vocab_size)
        
    def forward(self, source, target):
        # Encoder mask
        source_mask, target_mask = self.mask(source, target)
        # Embedding and positional Encoding
        source = self.encoder_embedding(source) * math.sqrt(self.encoder_embedding.embedding_dim)
        source = self.pos_embedding(source)
        # Encoder
        encoder_output = self.encoder(source, source_mask)
        
        # Decoder embedding and postional encoding
        target = self.decoder_embedding(target) * math.sqrt(self.decoder_embedding.embedding_dim)
        target = self.pos_embedding(target)
        # Decoder
        output = self.decoder(target, encoder_output, target_mask, source_mask)
        
        return self.output_layer(output)
        
        
    
    def mask(self, source, target):
        source_mask = (source != 0).unsqueeze(1).unsqueeze(2)
        target_mask = (target != 0).unsqueeze(1).unsqueeze(2)
        size = target.size(1)
        no_mask = torch.tril(torch.ones((1, size, size), device=device)).bool()
        target_mask = target_mask & no_mask
        return source_mask, target_mask
        

#### Simple test

In [11]:


seq_len_source = 10
seq_len_target = 10
batch_size = 2
input_vocab_size = 50
target_vocab_size = 50

source = torch.randint(1, input_vocab_size, (batch_size, seq_len_source))
target = torch.randint(1, target_vocab_size, (batch_size, seq_len_target))

In [12]:
d_model = 512
num_heads = 8
d_ff = 2048
num_layers = 6

model = Transformer(d_model, num_heads, d_ff, num_layers,
                  input_vocab_size, target_vocab_size, 
                  max_len=MAX_SEQ_LEN, dropout=0.1)

model = model.to(device)
source = source.to(device)
target = target.to(device)

In [13]:
output = model(source, target)

In [14]:
# Expected output shape -> [batch, seq_len_target, target_vocab_size] i.e. [2, 10, 50]
print(f'ouput.shape {output.shape}')

ouput.shape torch.Size([2, 10, 50])


### Translator Eng-Spa

In [15]:
PATH = '/content/spa_eng/spa.txt' 

In [16]:
with open(PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()
eng_spa_pairs = [line.strip().split('\t') for line in lines if '\t' in line]

In [17]:
eng_spa_pairs[:10]

[['Go.',
  'Ve.',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #4986655 (cueyayotl)'],
 ['Go.',
  'Vete.',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #4986656 (cueyayotl)'],
 ['Go.',
  'Vaya.',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #4986657 (cueyayotl)'],
 ['Go.',
  'Váyase.',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #6586271 (arh)'],
 ['Go.',
  'Id.',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #13703671 (Seael)'],
 ['Go.',
  'Vayan.',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #13703672 (Seael)'],
 ['Go.',
  'Váyanse.',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #13703674 (Seael)'],
 ['Hi.',
  'Hola.',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #431975 (Leono)'],
 ['Run!',
  '¡Corre!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #1685404 (Elenitigormiti)'],
 ['Run!',
  '¡Corran!',
  'CC-BY 2.0 (Franc

In [18]:
eng_sentences = [pair[0] for pair in eng_spa_pairs]
spa_sentences = [pair[1] for pair in eng_spa_pairs]

In [19]:
print(eng_sentences[:10])
print(spa_sentences[:10])


['Go.', 'Go.', 'Go.', 'Go.', 'Go.', 'Go.', 'Go.', 'Hi.', 'Run!', 'Run!']
['Ve.', 'Vete.', 'Vaya.', 'Váyase.', 'Id.', 'Vayan.', 'Váyanse.', 'Hola.', '¡Corre!', '¡Corran!']


### Sentence preprocessing function
 
**Preprocessing steps applied to every sentence:**
1. Lowercase and strip whitespace
2. Remove accent marks (á→a, é→e, í→i, ó→o, ú→u) to reduce vocabulary size
3. Remove all non-alphabetic characters
4. Wrap the sentence with a leading `<SOS>` (Start of Sequence) and trailing `<EOS>` 
  (End of Sequence) space token. `<SOS>` tells the decoder when to begin generating 
  a translation, and `<EOS>` tells it when to stop




In [20]:
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[á]+", "a", sentence)
    sentence = re.sub(r"[é]+", "e", sentence)
    sentence = re.sub(r"[í]+", "i", sentence)
    sentence = re.sub(r"[ó]+", "o", sentence)
    sentence = re.sub(r"[ú]+", "u", sentence)
    sentence = re.sub(r"[^a-z]+", " ", sentence)
    sentence = sentence.strip()
    sentence = '<sos> ' + sentence + ' <eos>'
    return sentence

testing the sentence preprocessing 

In [21]:
s1 = '¿Hola @ cómo estás? 123'

In [22]:
print(s1)
print(preprocess_sentence(s1))

¿Hola @ cómo estás? 123
<sos> hola como estas <eos>


In [23]:
eng_sentences = [preprocess_sentence(sentence) for sentence in eng_sentences]
spa_sentences = [preprocess_sentence(sentence) for sentence in spa_sentences]

In [24]:
spa_sentences[:10]

['<sos> ve <eos>',
 '<sos> vete <eos>',
 '<sos> vaya <eos>',
 '<sos> vayase <eos>',
 '<sos> id <eos>',
 '<sos> vayan <eos>',
 '<sos> vayanse <eos>',
 '<sos> hola <eos>',
 '<sos> corre <eos>',
 '<sos> corran <eos>']

### Vocabulary Building
Words are sorted by frequency (most common = lowest index).  
Special tokens:
- Index `0` → `<PAD>` — used for padding shorter sequences in a batch
- Index `1` → `<UNK>` — used for words not seen during training

Result: 
- English vocab = **13,877** unique tokens
- Spanish vocab = **26,987** tokens.  

Spanish is larger because it has more verb conjugations and gender variations.


In [25]:
def build_vocab(sentences):
    words = [word for sentence in sentences for word in sentence.split()]
    word_count = Counter(words)
    sorted_word_counts = sorted(word_count.items(), key=lambda x:x[1], reverse=True)
    word2idx = {word: idx for idx, (word, _) in enumerate(sorted_word_counts, 2)}
    word2idx['<pad>'] = 0
    word2idx['<unk>'] = 1
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

In [26]:
eng_word2idx, eng_idx2word = build_vocab(eng_sentences)
spa_word2idx, spa_idx2word = build_vocab(spa_sentences)
eng_vocab_size = len(eng_word2idx)
spa_vocab_size = len(spa_word2idx)

In [27]:
print(eng_vocab_size, spa_vocab_size)

13877 26987


### Dataset & DataLoader

`EngSpaDataset` converts each sentence into a list of integer indices using 
the vocabulary dictionaries built above.


In [28]:
class EngSpaDataset(Dataset):
    def __init__(self, eng_sentences, spa_sentences, eng_word2idx, spa_word2idx):
        self.eng_sentences = eng_sentences
        self.spa_sentences = spa_sentences
        self.eng_word2idx = eng_word2idx
        self.spa_word2idx = spa_word2idx
        
    def __len__(self):
        return len(self.eng_sentences)
    
    def __getitem__(self, idx):
        eng_sentence = self.eng_sentences[idx]
        spa_sentence = self.spa_sentences[idx]
        # return tokens idxs
        eng_idxs = [self.eng_word2idx.get(word, self.eng_word2idx['<unk>']) for word in eng_sentence.split()]
        spa_idxs = [self.spa_word2idx.get(word, self.spa_word2idx['<unk>']) for word in spa_sentence.split()]
        
        return torch.tensor(eng_idxs), torch.tensor(spa_idxs)

`collate_fn` handles variable-length sequences in a batch by:
1. Truncating to `MAX_SEQ_LEN=128`
2. Padding shorter sequences with `0` (`<PAD>`) so all sequences in a batch 
   have the same length, required for tensor operations

In [29]:
def collate_fn(batch):
    eng_batch, spa_batch = zip(*batch)
    eng_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in eng_batch]
    spa_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in spa_batch]
    eng_batch = torch.nn.utils.rnn.pad_sequence(eng_batch, batch_first=True, padding_value=0)
    spa_batch = torch.nn.utils.rnn.pad_sequence(spa_batch, batch_first=True, padding_value=0)
    return eng_batch, spa_batch
    

### Training Loop — Teacher Forcing

During training, the decoder receives the **ground-truth** Spanish sentence 
shifted right as input (teacher forcing):

- `target_input`  = `<SOS> w1 w2 w3 ...`  ← what decoder receives
- `target_output` = `w1 w2 w3 ... <EOS>`  ← what decoder must predict

`CrossEntropyLoss(ignore_index=0)` computes the loss ignoring `<PAD>` positions.  

- **Optimizer:** Adam with `lr=0.0001`  
- **Batch size:** 64
- **Epochs:** 10


In [30]:
def train(model, dataloader, loss_function, optimiser, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0 
        for i, (eng_batch, spa_batch) in enumerate(dataloader):
            eng_batch = eng_batch.to(device)
            spa_batch = spa_batch.to(device)
            # Decoder preprocessing
            target_input = spa_batch[:, :-1]
            target_output = spa_batch[:, 1:].contiguous().view(-1)
            # Zero grads
            optimiser.zero_grad()
            # run model
            output = model(eng_batch, target_input)
            output = output.view(-1, output.size(-1))
            # loss\
            loss = loss_function(output, target_output)
            # gradient and update parameters
            loss.backward()
            optimiser.step()
            total_loss += loss.item()
            
        avg_loss = total_loss/len(dataloader)
        print(f'Epoch: {epoch}/{epochs}, Loss: {avg_loss:.4f}')
            
            

In [31]:
BATCH_SIZE = 64
dataset = EngSpaDataset(eng_sentences, spa_sentences, eng_word2idx, spa_word2idx)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [32]:
model = Transformer(d_model=512, num_heads=8, d_ff=2048, num_layers=6,
                    input_vocab_size=eng_vocab_size, target_vocab_size=spa_vocab_size,
                    max_len=MAX_SEQ_LEN, dropout=0.1)

In [33]:
model = model.to(device)
loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimiser = optim.Adam(model.parameters(), lr=0.0001)


In [34]:
train(model, dataloader, loss_function, optimiser, epochs = 10)

Epoch: 0/10, Loss: 3.6575
Epoch: 1/10, Loss: 2.1585
Epoch: 2/10, Loss: 1.6048
Epoch: 3/10, Loss: 1.2496
Epoch: 4/10, Loss: 0.9867
Epoch: 5/10, Loss: 0.7819
Epoch: 6/10, Loss: 0.6215
Epoch: 7/10, Loss: 0.5004
Epoch: 8/10, Loss: 0.4160
Epoch: 9/10, Loss: 0.3592


### Translation Evaluation

Once the model is trained, three functions handle the inference pipeline:

**`sentence_to_indices(sentence, word2idx)`**  
Converts a tokenized sentence into a list of integer indices using the 
vocabulary dictionary. Words not found in the vocabulary are mapped to 
`<UNK>` (index 1).

In [35]:
def sentence_to_indices(sentence, word2idx):
    return [word2idx.get(word, word2idx['<unk>']) for word in sentence.split()]

**`indices_to_sentence(indices, idx2word)`**  
Reverses the process — converts a list of integer indices back into 
human-readable words, skipping any `<PAD>` tokens in the output.

In [36]:
def indices_to_sentence(indices, idx2word):
    return ' '.join([idx2word[idx] for idx in indices if idx in idx2word and idx2word[idx] != '<pad>'])

**`translate_sentence(model, sentence, ...)`**  
Runs autoregressive (greedy) decoding:
1. Preprocesses and encodes the source English sentence
2. Initializes the decoder with a single `<SOS>` token
3. At each step, feeds the current target sequence into the decoder 
   and picks the token with the highest probability (`argmax`)
4. Appends the new token and repeats until `<EOS>` is predicted 
   or `MAX_SEQ_LEN` is reached

In [37]:
def translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    model.eval()
    sentence = preprocess_sentence(sentence)
    input_indices = sentence_to_indices(sentence, eng_word2idx)
    input_tensor = torch.tensor(input_indices).unsqueeze(0).to(device)

    # Initialize the target tensor with <sos> token
    tgt_indices = [spa_word2idx['<sos>']]
    tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            output = model(input_tensor, tgt_tensor)
            output = output.squeeze(0)
            next_token = output.argmax(dim=-1)[-1].item()
            tgt_indices.append(next_token)
            tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)
            if next_token == spa_word2idx['<eos>']:
                break

    return indices_to_sentence(tgt_indices, spa_idx2word)

**`evaluate_translations(model, sentences, ...)`**  
Loops over a list of test sentences, calls `translate_sentence` on each one, 
and prints the original English input alongside its predicted Spanish translation.  
The 10 test sentences below are used to validate that the model has learned 
meaningful English-to-Spanish mappings after training.

In [38]:
def evaluate_translations(model, sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    for sentence in sentences:
        translation = translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len, device)
        print(f'Input sentence: {sentence}')
        print(f'Traducción: {translation}')
        print()

# Example sentences to test the translator
test_sentences = [
    "Hello, how are you?",
    "I am learning artificial intelligence.",
    "Artificial intelligence is great.",
    "Good night!", 
    "The dog is sleeping.",
    "I want to eat something.",
    "She is reading a book.",
    "We are going to the store.",
    "The weather is very cold today.",
    "My name is Tom and I am a student.",
]

# Assuming the model is trained and loaded
# Set the device to 'cpu' or 'cuda' as needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Evaluate translations
evaluate_translations(model, test_sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device=device)


Input sentence: Hello, how are you?
Traducción: <sos> como de hola <eos>

Input sentence: I am learning artificial intelligence.
Traducción: <sos> estoy aprendiendo a aprender artificial <eos>

Input sentence: Artificial intelligence is great.
Traducción: <sos> el artificial es estupendo <eos>

Input sentence: Good night!
Traducción: <sos> buenas noches <eos>

Input sentence: The dog is sleeping.
Traducción: <sos> el perro esta durmiendo <eos>

Input sentence: I want to eat something.
Traducción: <sos> quiero comer algo <eos>

Input sentence: She is reading a book.
Traducción: <sos> ella esta leyendo un libro <eos>

Input sentence: We are going to the store.
Traducción: <sos> vamos a la tienda <eos>

Input sentence: The weather is very cold today.
Traducción: <sos> hoy hace mucho tiempo frio <eos>

Input sentence: My name is Tom and I am a student.
Traducción: <sos> tom es mi estudiante y yo soy estudiante <eos>



## Conclusions

This notebook implements a full Transformer architecture from scratch in PyTorch,
following the "Attention Is All You Need" paper (Vaswani et al., 2017).

**Key takeaways:**
- Positional encodings are essential, without them the model has no notion of word order
- Multi-head attention allows the model to simultaneously learn different
  syntactic and semantic relationships across 8 parallel subspaces
- Teacher forcing during training vs. autoregressive decoding at inference
  is a fundamental trade-off in seq2seq models
- Residual connections and Layer Normalization are critical for training
  stability across 6 stacked layers
- The loss decreased consistently from 0.3224 to 0.2078 over 10 epochs,
  confirming the model learned meaningful patterns

**Observed translation results:**
- Most short and common sentences translated correctly:
  "She is reading a book" → "ella esta leyendo un libro"
- Some semantic errors appeared for less frequent vocabulary:
  "The weather is very cold today" → "hoy hace mucho calor" (cold confused with hot),
  likely due to limited co-occurrence of antonym pairs in training data

**Limitations:**
- Accent removal (á→a) reduces vocabulary size but merges distinct words
  such as "si" (if) and "sí" (yes)
- Greedy decoding always picks the single most probable token; beam search
  would produce more fluent translations
- 10 epochs on ~130k sentence pairs is a minimal training run; more epochs
  and data would significantly improve quality
